# Notebook 08 — FFN capacity

Paired with `theory/08_feedforward.md`. **Mode:** theory-first.

## QHMPC

**Question.**
1. FFN-as-memory hand example: outputs match the §8.4 worked computation.
2. Parameter / FLOP accounting for vanilla vs SwiGLU at matched FLOPs.
3. Capacity sweep: train tiny transformer on a memorization task across $d_{\text{ff}} / d_{\text{model}} \in \{0.5, 1, 2, 4, 8\}$.
4. FFN ablation: zero the FFN, measure accuracy drop.
5. Activation ablation: GELU vs ReLU vs SiLU.

**Method.** Experiments 1–2 structural. 3–5 train tiny 1-layer transformer.

**Prediction.** *Paste §8.7 predictions.*

**Confidence.** *[low/medium/high]*.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

np.random.seed(0); torch.manual_seed(0)

## Experiment 1 — FFN as memory, hand example

In [ ]:
W1 = np.array([[1, 0], [0, 1], [1, 1]], dtype=float)    # (d_ff=3, d_model=2)
W2 = np.array([[1, 0, 0], [0, 1, 0]], dtype=float)      # (d_model=2, d_ff=3)

def ffn(x):
    return W2 @ np.maximum(W1 @ x, 0)

for x in [np.array([2., -1.]), np.array([-1., 2.]), np.array([-1., -1.]), np.array([1., 1.])]:
    print(f'x = {x}  ->  hidden = {np.maximum(W1 @ x, 0)}  ->  output = {ffn(x)}')

**Sub-finding 1.** *Outputs match §8.4 hand computation? Each input lights up which memory slots?*

## Experiment 2 — parameter & FLOP accounting (Prop 8.4)

In [ ]:
d_model = 768
for ff_mult in [1, 2, 4, 8]:
    d_ff = ff_mult * d_model
    vanilla_params = 2 * d_model * d_ff
    swiglu_params = 3 * d_model * d_ff
    vanilla_flops = 4 * d_model * d_ff
    swiglu_flops = 6 * d_model * d_ff
    print(f'd_ff = {ff_mult}*d_model = {d_ff:>5}:')
    print(f'  vanilla:  params = {vanilla_params:>10}   flops = {vanilla_flops:>10}')
    print(f'  SwiGLU:   params = {swiglu_params:>10}   flops = {swiglu_flops:>10}')

# SwiGLU at matched FLOPs to vanilla(d_ff=4*d_model): solve 6 d_ff_swi = 4 * 4 d_model => d_ff_swi = 8/3 d_model
print(f'\nSwiGLU at matched FLOPs to vanilla(4x): d_ff = 8/3 * d_model ≈ {(8/3)*d_model:.0f}')

**Sub-finding 2.** *SwiGLU at matched FLOPs uses $\sim 2.67 \times$ width vs vanilla's $4 \times$. The literature uses $d_{\text{ff}} \approx 2.67 d_{\text{model}}$ for SwiGLU — consistent.*

## Experiment 3 — capacity sweep on a memorization task

Tiny 1-layer transformer on a key→value retrieval task with $V = 32$ tokens. Sweep $d_{\text{ff}}$.

In [ ]:
V_key, V_val = 32, 32
n_pairs, seq_len = 5, 11
d_model = 32
V_total = V_key + V_val

def make_batch(B):
    keys = torch.randint(0, V_key, (B, n_pairs))
    vals = torch.randint(0, V_val, (B, n_pairs))
    qi = torch.randint(0, n_pairs, (B,))
    targets = vals[torch.arange(B), qi]
    seq = torch.zeros(B, seq_len, dtype=torch.long)
    for b in range(B):
        for p in range(n_pairs):
            seq[b, 2*p] = keys[b, p]
            seq[b, 2*p + 1] = V_key + vals[b, p]
        seq[b, -1] = keys[b, qi[b]]
    return seq, targets

class TinyT(nn.Module):
    def __init__(self, d_ff, ffn_off=False, activation='gelu'):
        super().__init__()
        self.embed = nn.Embedding(V_total, d_model)
        self.pos = nn.Parameter(torch.randn(seq_len, d_model) * 0.02)
        self.attn = nn.MultiheadAttention(d_model, num_heads=4, batch_first=True, bias=False)
        self.ln1 = nn.LayerNorm(d_model); self.ln2 = nn.LayerNorm(d_model)
        if ffn_off:
            self.ffn = nn.Identity()
        else:
            act = {'gelu': nn.GELU(), 'relu': nn.ReLU(), 'silu': nn.SiLU()}[activation]
            self.ffn = nn.Sequential(nn.Linear(d_model, d_ff), act, nn.Linear(d_ff, d_model))
        self.out = nn.Linear(d_model, V_val, bias=False)

    def forward(self, seq):
        x = self.embed(seq) + self.pos
        a, _ = self.attn(self.ln1(x), self.ln1(x), self.ln1(x), need_weights=False)
        x = x + a
        x = x + self.ffn(self.ln2(x))
        return self.out(x[:, -1])

def train_eval(d_ff, ffn_off=False, activation='gelu', steps=2000, lr=3e-3):
    torch.manual_seed(0)
    m = TinyT(d_ff, ffn_off=ffn_off, activation=activation)
    opt = torch.optim.Adam(m.parameters(), lr=lr)
    for _ in range(steps):
        seq, tgt = make_batch(64)
        logits = m(seq)
        loss = F.cross_entropy(logits, tgt)
        opt.zero_grad(); loss.backward(); opt.step()
    with torch.no_grad():
        seq, tgt = make_batch(2000)
        return (m(seq).argmax(-1) == tgt).float().mean().item()

widths = [16, 32, 64, 128, 256]
accs = []
for d_ff in widths:
    acc = train_eval(d_ff)
    accs.append(acc)
    print(f'd_ff = {d_ff:>4}  (d_ff/d_model = {d_ff/d_model:>5.2f}):  acc = {acc:.3f}')

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot([w/d_model for w in widths], accs, 'o-')
ax.set_xlabel('d_ff / d_model'); ax.set_ylabel('accuracy')
ax.set_title('FFN width vs. retrieval accuracy')
ax.set_xscale('log')
plt.tight_layout(); plt.show()

**Sub-finding 3.** *At what width did the model click? Diminishing returns above the threshold?*

## Experiment 4 — FFN ablation

In [ ]:
acc_with = train_eval(d_ff=128)
acc_without = train_eval(d_ff=128, ffn_off=True)
print(f'with FFN:    acc = {acc_with:.3f}')
print(f'without FFN: acc = {acc_without:.3f}    (chance = {1/V_val:.3f})')

**Sub-finding 4.** *Did removing the FFN drop accuracy to near chance, or did attention alone manage? What does this say about which sub-layer carries the retrieval logic?*

## Experiment 5 — activation ablation (GELU vs ReLU vs SiLU)

In [ ]:
for act in ['gelu', 'relu', 'silu']:
    acc = train_eval(d_ff=128, activation=act)
    print(f'activation = {act:>5}: acc = {acc:.3f}')

**Sub-finding 5.** *Material difference, or noise-level? Confirms the literature claim that activation choice is one of the smaller hyperparameter levers.*

## Finding

*3–6 sentences. Did the FFN turn out to be the load-bearing piece for memorization? Did the capacity threshold match the per-task `d_ff` you'd expect from $V$ memory cells?*